In [7]:
import pandas as pd
import numpy as np
import itertools
from tqdm import tqdm

In [10]:
print("Starting Advanced GNN-Specific Non-IID Partitioning...")

# Load the engineered data
try:
    df = pd.read_parquet("../data/processed/train_engineered.parquet")
except FileNotFoundError:
    df = pd.read_csv("../data/processed/train_engineered.csv")

# ---------------------------------------------------------
# 1. BANK 4: Small Credit Union (Temporal Skew)
# We sort by Time (TransactionDT) and slice the earliest 5%
# ---------------------------------------------------------
df = df.sort_values('TransactionDT').copy()
n_bank4 = int(len(df) * 0.05)

client_4_df = df.iloc[:n_bank4].copy()
remaining_df = df.iloc[n_bank4:].copy()

# Split remaining pool into fraud and legit to precisely control the other banks
fraud_pool = remaining_df[remaining_df['isFraud'] == 1].copy()
legit_pool = remaining_df[remaining_df['isFraud'] == 0].copy()

# ---------------------------------------------------------
# 2. BANK 2: High-Risk Neobank (Label Skew)
# Force ~65% of the TOTAL dataset's fraud into this single client
# ---------------------------------------------------------
target_fraud_b2 = int(len(df[df['isFraud'] == 1]) * 0.65)
n_fraud_b2 = min(target_fraud_b2, len(fraud_pool))

b2_fraud = fraud_pool.sample(n=n_fraud_b2, random_state=42)
# Give them a relatively small amount of legit data to make the fraud dense (~40% fraud rate)
b2_legit = legit_pool.sample(n=20000, random_state=42) 
client_2_df = pd.concat([b2_fraud, b2_legit]).sample(frac=1)

# Remove used rows
fraud_pool = fraud_pool.drop(b2_fraud.index)
legit_pool = legit_pool.drop(b2_legit.index)

# ---------------------------------------------------------
# 3. BANK 3: Wealth Management (Feature + Label Skew)
# High Transaction Amount only
# ---------------------------------------------------------
# Find the 80th percentile of the log-amount in the remaining pool
high_amt_thresh = legit_pool['TransactionAmt_Log'].quantile(0.80)

b3_legit_pool = legit_pool[legit_pool['TransactionAmt_Log'] > high_amt_thresh]
b3_fraud_pool = fraud_pool[fraud_pool['TransactionAmt_Log'] > high_amt_thresh]

b3_legit = b3_legit_pool.sample(n=min(35000, len(b3_legit_pool)), random_state=42)
b3_fraud = b3_fraud_pool.sample(n=min(1000, len(b3_fraud_pool)), random_state=42)
client_3_df = pd.concat([b3_fraud, b3_legit]).sample(frac=1)

fraud_pool = fraud_pool.drop(b3_fraud.index)
legit_pool = legit_pool.drop(b3_legit.index)

# ---------------------------------------------------------
# 4. BANK 1: Regional Retail Bank (Feature Skew)
# Filter by Region (addr1) AND Low Transaction Amount
# ---------------------------------------------------------
median_addr = legit_pool['addr1'].median()
low_amt_thresh = legit_pool['TransactionAmt_Log'].quantile(0.50)

b1_legit_pool = legit_pool[(legit_pool['addr1'] > median_addr) & (legit_pool['TransactionAmt_Log'] <= low_amt_thresh)]
b1_fraud_pool = fraud_pool[(fraud_pool['addr1'] > median_addr) & (fraud_pool['TransactionAmt_Log'] <= low_amt_thresh)]

b1_legit = b1_legit_pool.sample(n=min(40000, len(b1_legit_pool)), random_state=42)
b1_fraud = b1_fraud_pool.sample(n=min(1000, len(b1_fraud_pool)), random_state=42)
client_1_df = pd.concat([b1_fraud, b1_legit]).sample(frac=1)

fraud_pool = fraud_pool.drop(b1_fraud.index)
legit_pool = legit_pool.drop(b1_legit.index)

# ---------------------------------------------------------
# 5. BANK 0: Global Mega-Bank (Quantity Skew)
# Gets the entire remaining backbone of the dataset (~50%)
# ---------------------------------------------------------
client_0_df = pd.concat([fraud_pool, legit_pool]).sample(frac=1)

# --- SUMMARY STATISTICS ---
client_dfs = {
    'Bank_0_Mega': client_0_df,
    'Bank_1_Regional': client_1_df,
    'Bank_2_HighRisk': client_2_df,
    'Bank_3_Wealth': client_3_df,
    'Bank_4_CreditUnion': client_4_df
}

print("\n--- Advanced GNN Partitioning Complete ---")
for name, c_df in client_dfs.items():
    fraud_rate = c_df['isFraud'].mean() * 100
    print(f"{name:<22} | Nodes: {len(c_df):>7,} | Fraud Rate: {fraud_rate:>5.2f}%")

Starting Advanced GNN-Specific Non-IID Partitioning...

--- Advanced GNN Partitioning Complete ---
Bank_0_Mega            | Nodes: 450,960 | Fraud Rate:  1.06%
Bank_1_Regional        | Nodes:  40,623 | Fraud Rate:  1.53%
Bank_2_HighRisk        | Nodes:  33,430 | Fraud Rate: 40.17%
Bank_3_Wealth          | Nodes:  36,000 | Fraud Rate:  2.78%
Bank_4_CreditUnion     | Nodes:  29,527 | Fraud Rate:  2.88%


In [9]:
df.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V_PCA_66,V_PCA_67,V_PCA_68,V_PCA_69,V_PCA_70,V_PCA_71,V_PCA_72,V_PCA_73,V_PCA_74,V_PCA_75
0,2987000,0,86400,-0.278167,0.547250,13926,361.0,150.0,-2.607805,142.0,...,0.052138,0.230623,-0.024913,0.002747,0.371474,-0.193850,-0.029408,-0.120300,0.378421,-0.941987
1,2987001,0,86401,-0.443327,0.547250,2755,404.0,150.0,-0.982065,102.0,...,0.186519,0.046598,-0.019032,0.056693,0.043953,-0.367579,-0.039060,-0.155186,-0.029928,0.028537
2,2987002,0,86469,-0.317889,0.547250,4663,490.0,150.0,0.643675,166.0,...,-0.045728,0.177337,-0.204153,-0.043263,-0.107568,-0.137679,0.041012,-0.049010,0.056597,0.000791
3,2987003,0,86499,-0.355521,0.547250,18132,567.0,150.0,-0.982065,117.0,...,-0.205796,-0.488660,-0.270685,-0.037135,-0.018018,0.349232,-0.082465,-0.070597,-0.568015,0.163743
4,2987004,0,86506,-0.355521,-1.559603,4497,514.0,150.0,-0.982065,102.0,...,-0.135637,0.151019,0.271342,-0.084615,-0.030807,-0.049905,-0.079344,-0.003563,0.197975,0.179341


In [11]:
print("Starting Graph Topology Construction...")


# ---------------------------------------------------------
# STEP 1: SHADOW NODE DETECTOR (Global Intersection)
# ---------------------------------------------------------
print("\n--- Running Global Shadow Node Detector ---")
# We will use 'card1' as our primary entity to link fraud rings across banks
client_card_sets = {}

# Extract the unique cards present in each client
for name, df_client in client_dfs.items():
    client_card_sets[name] = set(df_client['card1'].dropna().unique())

# Find cards that exist in MORE THAN ONE client
global_card_counts = {}
for name, card_set in client_card_sets.items():
    for card in card_set:
        global_card_counts[card] = global_card_counts.get(card, 0) + 1

# If a card is in 2 or more banks, it is a border entity (Shadow Entity)
shadow_entities = {card for card, count in global_card_counts.items() if count > 1}
print(f"Discovered {len(shadow_entities):,} unique Shadow Entities (Cards shared across banks).")

# Flag the specific transactions in each client that involve a Shadow Entity
for name, df_client in client_dfs.items():
    # Create a boolean mask: True if the transaction's card1 is in the shadow_entities set
    df_client['is_shadow_node'] = df_client['card1'].isin(shadow_entities)
    num_shadows = df_client['is_shadow_node'].sum()
    print(f"{name:<20}: {num_shadows:>6,} Shadow Nodes identified.")


# ---------------------------------------------------------
# STEP 2: LOCAL EDGE BUILDER (Chronological Connections)
# ---------------------------------------------------------
print("\n--- Building Local Client Edges (Sequential) ---")

def build_sequential_edges(df, entity_col='card1'):
    """Builds a chronological edge_index for transactions sharing a specific entity."""
    
    # 1. CRITICAL: Sort by Time before resetting index!
    # This guarantees that Row 0 happens before Row 1
    df_sorted = df.sort_values('TransactionDT').reset_index(drop=True)
    
    # Group the row indices by the entity ID
    grouped_nodes = df_sorted.groupby(entity_col).indices
    
    source_nodes = []
    target_nodes = []
    
    # Connect transactions chronologically (1->2, 2->3)
    for entity_id, node_indices in grouped_nodes.items():
        # node_indices is naturally sorted by time because of step 1
        if len(node_indices) > 1:
            for i in range(len(node_indices) - 1):
                u = node_indices[i]      # Current transaction
                v = node_indices[i + 1]  # Next transaction
                
                # Add bidirectional edge
                source_nodes.extend([u, v])
                target_nodes.extend([v, u])
                
    # Use int32 to save RAM (Safe up to 2 Billion edges)
    edge_index = np.array([source_nodes, target_nodes], dtype=np.int32)
    return edge_index, df_sorted

client_edges = {}
client_dfs_reset = {} 

for name, df_client in client_dfs.items():
    edges, df_reset = build_sequential_edges(df_client, entity_col='card1')
    client_edges[name] = edges
    client_dfs_reset[name] = df_reset
    print(f"{name:<20}: Generated {edges.shape[1]:>10,} chronological edges.")

print("\nGraph Topology successfully constructed and memory-optimized!")

Starting Graph Topology Construction...

--- Running Global Shadow Node Detector ---
Discovered 7,300 unique Shadow Entities (Cards shared across banks).
Bank_0_Mega         : 436,769 Shadow Nodes identified.
Bank_1_Regional     : 40,401 Shadow Nodes identified.
Bank_2_HighRisk     : 33,193 Shadow Nodes identified.
Bank_3_Wealth       : 35,703 Shadow Nodes identified.
Bank_4_CreditUnion  : 29,197 Shadow Nodes identified.

--- Building Local Client Edges (Sequential) ---
Bank_0_Mega         : Generated    876,916 chronological edges.
Bank_1_Regional     : Generated     74,464 chronological edges.
Bank_2_HighRisk     : Generated     58,482 chronological edges.
Bank_3_Wealth       : Generated     62,870 chronological edges.
Bank_4_CreditUnion  : Generated     50,932 chronological edges.

Graph Topology successfully constructed and memory-optimized!
